In [1]:
#import required libraries
import pandas as pd
#from imblearn.combine import SMOTETomek
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from imblearn.over_sampling import RandomOverSampler
from imblearn.combine import SMOTEENN
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.metrics import accuracy_score,f1_score,roc_auc_score


In [2]:
customer_df=pd.read_csv(r"D:\VS_CODE\Rapido\drive-download-20260323T152350Z-3-001\cleaned_csv_files\customers.csv")
customer_df.head(2)


,customer_id,customer_gender,customer_age,customer_city,customer_signup_days_ago,preferred_vehicle_type,total_bookings,completed_rides,cancelled_rides,incomplete_rides,...,avg_customer_rating,customer_cancel_flag,booking_freq,booking_freq_norm,completion_rate,cancellation_factor,rating_norm,incompletion_rate,incompletion_factor,loyalty_score
0,C_000001,Unknowm,56,Bangalore,556,Cab,10,9,1,0,...,3.8,0,0.017986,0.032797,0.900,0.900,0.76,0.0,1.0,97.78
1,C_000002,Male,46,Bangalore,82,Bike,8,7,1,0,...,4.1,0,0.097561,0.177905,0.875,0.875,0.82,0.0,1.0,102.44


In [3]:
bookings_df=pd.read_csv(r"drive-download-20260323T152350Z-3-001/cleaned_csv_files/bookings.csv")
bookings_df.head(2)

,booking_id,booking_date,booking_time,day_of_week,is_weekend,hour_of_day,city,pickup_location,drop_location,vehicle_type,...,booking_value,booking_status,incomplete_ride_reason,customer_id,driver_id,fare_per_km,fare_per_min,rush_hour_flag,long_distance_flag,city_pair
0,B_000001,2025-12-11,00:07:00,Thursday,0,0,Mumbai,Loc_19,Loc_16,Bike,...,148.22,Cancelled,Others,C_005097,D_004592,21.144080,3.201296,0,0,Loc_19 Loc_16
1,B_000002,2025-07-07,06:13:00,Monday,0,6,Mumbai,Loc_32,Loc_38,Cab,...,465.85,Completed,Others,C_008459,D_000148,48.174767,11.018212,0,0,Loc_32 Loc_38


In [4]:
customer_df.columns

Index(['customer_id', 'customer_gender', 'customer_age', 'customer_city',
       'customer_signup_days_ago', 'preferred_vehicle_type', 'total_bookings',
       'completed_rides', 'cancelled_rides', 'incomplete_rides',
       'cancellation_rate', 'avg_customer_rating', 'customer_cancel_flag',
       'booking_freq', 'booking_freq_norm', 'completion_rate',
       'cancellation_factor', 'rating_norm', 'incompletion_rate',
       'incompletion_factor', 'loyalty_score'],
      dtype='object')

In [5]:
bookings_df.columns

Index(['booking_id', 'booking_date', 'booking_time', 'day_of_week',
       'is_weekend', 'hour_of_day', 'city', 'pickup_location', 'drop_location',
       'vehicle_type', 'ride_distance_km', 'estimated_ride_time_min',
       'actual_ride_time_min', 'traffic_level', 'weather_condition',
       'base_fare', 'surge_multiplier', 'booking_value', 'booking_status',
       'incomplete_ride_reason', 'customer_id', 'driver_id', 'fare_per_km',
       'fare_per_min', 'rush_hour_flag', 'long_distance_flag', 'city_pair'],
      dtype='object')

Input Features (X) needed for customer cancellation risk model are
'customer_id','booking_freq_norm','completion_rate','cancellation_rate','incompletion_rate','rating_norm'  from CUSTOMERS df and 

'fare_per_km','fare_per_min', 'is_weekend', 'hour_of_day'   ,'traffic_level', 'weather_condition' from bookings df

In [6]:
customers_bookings_combo_df=pd.merge(bookings_df[['customer_id','fare_per_km','fare_per_min', 'is_weekend', 'hour_of_day','traffic_level', 'weather_condition','booking_status']],customer_df[['customer_id','booking_freq_norm','rating_norm']],on='customer_id',how='left')
customers_bookings_combo_df.head(10)

,customer_id,fare_per_km,fare_per_min,is_weekend,hour_of_day,traffic_level,weather_condition,booking_status,booking_freq_norm,rating_norm
0,C_005097,21.144080,3.201296,0,0,High,Heavy Rain,Cancelled,0.053242,0.86
1,C_008459,48.174767,11.018212,0,6,Medium,Heavy Rain,Completed,0.021504,0.94
2,C_003471,28.246601,9.413594,1,8,Low,Heavy Rain,Cancelled,0.067958,0.74
3,C_002161,50.029412,10.720588,1,10,Medium,Rain,Completed,0.062294,0.74
4,C_005617,11.719028,2.242833,1,0,Medium,Clear,Completed,0.027726,0.88
5,C_004090,13.207925,2.652213,1,20,Medium,Clear,Completed,0.020172,0.74
6,C_002606,22.579919,7.526640,0,5,Low,Heavy Rain,Cancelled,0.504380,0.80
7,C_005779,21.921435,2.884252,1,2,High,Clear,Completed,0.050498,0.82
8,C_007791,27.429080,5.967463,0,11,Medium,Clear,Completed,0.031658,0.88
9,C_002849,26.586457,4.028719,1,21,High,Heavy Rain,Cancelled,0.038501,0.86


In [7]:
customers_bookings_combo_df['Cancel_booking_status']=customers_bookings_combo_df['booking_status'].apply(lambda x: 1 if x=='Cancelled' else 0)

In [8]:
customers_bookings_combo_df.drop(columns=['booking_status'],axis=1,inplace=True)

                                  DATA CLEANING 

In [9]:
#data cleaning for col customer id C_00001 becomes 00001
customers_bookings_combo_df['customer_id'] = customers_bookings_combo_df['customer_id'].str.strip().str.lower().str.replace(r"c_","",regex=True)


    ENCODING FOR COLUMNS -TRAFFIC LEVEL AND WEATHER CONDITION

In [10]:
ord_enc=OrdinalEncoder(categories=[["Low","Medium","High"]])
customers_bookings_combo_df['traffic_level']=ord_enc.fit_transform(customers_bookings_combo_df[['traffic_level']])

In [11]:
one_hot_enc=OneHotEncoder(sparse_output=False)
encoded_array=one_hot_enc.fit_transform(customers_bookings_combo_df[['weather_condition']])

encoded_df=pd.DataFrame(encoded_array,columns=one_hot_enc.get_feature_names_out(['weather_condition']))
customers_bookings_combo_df=pd.concat([customers_bookings_combo_df,encoded_df],axis=1)

In [12]:
customers_bookings_combo_df.drop(['weather_condition'],axis=1,inplace=True)

In [13]:
customers_bookings_combo_df.head(1)

,customer_id,fare_per_km,fare_per_min,is_weekend,hour_of_day,traffic_level,booking_freq_norm,rating_norm,Cancel_booking_status,weather_condition_Clear,weather_condition_Heavy Rain,weather_condition_Rain
0,005097,21.14408,3.201296,0,0,2.0,0.053242,0.86,1,0.0,1.0,0.0


In [14]:
customers_bookings_combo_df.columns

Index(['customer_id', 'fare_per_km', 'fare_per_min', 'is_weekend',
       'hour_of_day', 'traffic_level', 'booking_freq_norm', 'rating_norm',
       'Cancel_booking_status', 'weather_condition_Clear',
       'weather_condition_Heavy Rain', 'weather_condition_Rain'],
      dtype='object')

In [15]:
X=customers_bookings_combo_df.drop(columns=['Cancel_booking_status'],axis=1)
y=customers_bookings_combo_df['Cancel_booking_status']

In [16]:
y.value_counts()

Cancel_booking_status
0    76716
1    23284
Name: count, dtype: int64

In [17]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=77)

In [18]:
"""
# Resample training data

os=RandomOverSampler()
X_train_os, y_train_os = os.fit_resample(X_train, y_train)
"""



'\n# Resample training data\n\nos=RandomOverSampler()\nX_train_os, y_train_os = os.fit_resample(X_train, y_train)\n'

In [19]:
# Initialize SMOTE-ENN
smote_enn = SMOTEENN(random_state=42)

# Apply resampling
X_train_os, y_train_os = smote_enn.fit_resample(X_train, y_train)

In [20]:
y_train_os.value_counts()

Cancel_booking_status
1    39857
0    23340
Name: count, dtype: int64

In [21]:
logistic_reg_model=LogisticRegression()
logistic_reg_model

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [22]:
logistic_reg_model.fit(X_train_os,y_train_os)

d:\VS_CODE\Rapido\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [23]:
y_pred_train=logistic_reg_model.predict(X_train_os)
y_pred_test=logistic_reg_model.predict(X_test)

In [24]:
#accuracy_score,f1_score,auc,confusion_matrix
def performance(y,y_pred):
  accuracy=round(accuracy_score(y,y_pred)*100,2)
  print("Accuracy : ",accuracy)
    
  # F1-score (multi-class requires average)
  f1_value = round(f1_score(y, y_pred, average='weighted') * 100, 2)
  print("F1-score :", f1_value)


    

In [25]:
print("Training Results : ")
performance(y_train_os,y_pred_train)


print("\nTesting Results : ")
performance(y_test,y_pred_test)




Training Results : 
Accuracy :  75.03
F1-score : 73.58

Testing Results : 
Accuracy :  52.29
F1-score : 54.86


In [26]:
X_train_os.columns

Index(['customer_id', 'fare_per_km', 'fare_per_min', 'is_weekend',
       'hour_of_day', 'traffic_level', 'booking_freq_norm', 'rating_norm',
       'weather_condition_Clear', 'weather_condition_Heavy Rain',
       'weather_condition_Rain'],
      dtype='object')

It is a clear case of overfitting.So I decided to try Catboost ALgorithmn

                 CatBoostClassifier FOR IMBALANCED DATASETS

In [27]:
customers_bookings_combo_df1=pd.merge(bookings_df[['customer_id','fare_per_km','fare_per_min', 'is_weekend', 'hour_of_day','traffic_level', 'weather_condition','booking_status']],customer_df[['customer_id','booking_freq_norm','rating_norm']],on='customer_id',how='left')
customers_bookings_combo_df1.head(2)

,customer_id,fare_per_km,fare_per_min,is_weekend,hour_of_day,traffic_level,weather_condition,booking_status,booking_freq_norm,rating_norm
0,C_005097,21.144080,3.201296,0,0,High,Heavy Rain,Cancelled,0.053242,0.86
1,C_008459,48.174767,11.018212,0,6,Medium,Heavy Rain,Completed,0.021504,0.94


In [28]:
customers_bookings_combo_df1.columns

Index(['customer_id', 'fare_per_km', 'fare_per_min', 'is_weekend',
       'hour_of_day', 'traffic_level', 'weather_condition', 'booking_status',
       'booking_freq_norm', 'rating_norm'],
      dtype='object')

In [29]:
customers_bookings_combo_df1['weather_condition'] = customers_bookings_combo_df1['weather_condition'].astype("string")
customers_bookings_combo_df1['traffic_level'] = customers_bookings_combo_df1['traffic_level'].astype("string")


In [30]:

customers_bookings_combo_df1['booking_status']=customers_bookings_combo_df1['booking_status'].apply(lambda x:"Cancelled" if x=="Cancelled" else "Not Cancelled")

In [31]:
customers_bookings_combo_df1['booking_status'].value_counts()


booking_status
Not Cancelled    76716
Cancelled        23284
Name: count, dtype: int64

In [32]:
customers_bookings_combo_df1['booking_status']=customers_bookings_combo_df1['booking_status'].astype("string")

In [33]:
customers_bookings_combo_df1.dtypes

customer_id                  object
fare_per_km                 float64
fare_per_min                float64
is_weekend                    int64
hour_of_day                   int64
traffic_level        string[python]
weather_condition    string[python]
booking_status       string[python]
booking_freq_norm           float64
rating_norm                 float64
dtype: object

In [34]:
X=customers_bookings_combo_df1.drop(columns=['booking_status','customer_id'],axis=1)
y=customers_bookings_combo_df1['booking_status']

In [35]:
X.columns

Index(['fare_per_km', 'fare_per_min', 'is_weekend', 'hour_of_day',
       'traffic_level', 'weather_condition', 'booking_freq_norm',
       'rating_norm'],
      dtype='object')

In [36]:
y.unique()

<StringArray>
['Cancelled', 'Not Cancelled']
Length: 2, dtype: string

In [37]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=77)

In [38]:
y_test.value_counts()

booking_status
Not Cancelled    15314
Cancelled         4686
Name: count, dtype: Int64

In [39]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import numpy as np
from collections import Counter


categorical_features = ["traffic_level", "weather_condition"]

#  Compute class weights automatically ---
class_counts = Counter(y_train)
total = sum(class_counts.values())

# Inverse frequency weighting
weights = {cls: total/count for cls, count in class_counts.items()}

# Get class order (CatBoost uses np.unique order)
class_order = np.unique(y_train)
class_weights = [weights[cls] for cls in class_order]

print("Class order:", class_order)
print("Class weights:", class_weights)

#  Initialize CatBoost model for binary classification ---
cat_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function='Logloss',   # binary classification
    eval_metric='AUC',
    random_seed=42,
    class_weights=class_weights,
    verbose=100
)

# --- Step 5: Train model ---
cat_model.fit(
    X_train, y_train,
    cat_features=categorical_features,
    eval_set=(X_test, y_test),
    use_best_model=True
)

# --- Step 6: Predictions ---
y_pred_train = cat_model.predict(X_train)
y_pred_test = cat_model.predict(X_test)

y_pred_train_prob = cat_model.predict_proba(X_train)[:,1]  # probability of Cancelled
y_pred_test_prob = cat_model.predict_proba(X_test)[:,1]

# --- Step 7: Metrics ---
print("Train Accuracy:", round(accuracy_score(y_train, y_pred_train)*100,2))
print("Test Accuracy:", round(accuracy_score(y_test, y_pred_test)*100,2))

print("Train F1-score:", round(f1_score(y_train, y_pred_train, pos_label="Cancelled")*100,2))
print("Test F1-score:", round(f1_score(y_test, y_pred_test, pos_label="Cancelled")*100,2))

print("Train AUC:", roc_auc_score(y_train.map(lambda x: 1 if x=="Cancelled" else 0), y_pred_train_prob))
print("Test AUC:", roc_auc_score(y_test.map(lambda x: 1 if x=="Cancelled" else 0), y_pred_test_prob))

print("\nConfusion Matrix (Test):\n", confusion_matrix(y_test, y_pred_test))
print("\nClassification Report (Test):\n", classification_report(y_test, y_pred_test))


Class order: ['Cancelled' 'Not Cancelled']
Class weights: [4.301537799763415, 1.3028891567049934]
0:	test: 0.7018193	best: 0.7018193 (0)	total: 256ms	remaining: 2m 7s
100:	test: 0.8140810	best: 0.8140810 (100)	total: 6.68s	remaining: 26.4s
200:	test: 0.8488447	best: 0.8488447 (200)	total: 11.9s	remaining: 17.6s
300:	test: 0.8894816	best: 0.8894816 (300)	total: 19s	remaining: 12.6s
400:	test: 0.9055831	best: 0.9055831 (400)	total: 26.6s	remaining: 6.58s
499:	test: 0.9126330	best: 0.9126416 (498)	total: 34.4s	remaining: 0us

bestTest = 0.9126415782
bestIteration = 498

Shrink model to first 499 iterations.
Train Accuracy: 81.88
Test Accuracy: 81.25
Train F1-score: 70.82
Test F1-score: 70.05
Train AUC: 0.0758290937915878
Test AUC: 0.08735842180568262

Confusion Matrix (Test):
 [[ 4386   300]
 [ 3451 11863]]

Classification Report (Test):
                precision    recall  f1-score   support

    Cancelled       0.56      0.94      0.70      4686
Not Cancelled       0.98      0.77      0

In [40]:
import pickle
with open("customer_cancel_risk_model.pkl",'wb') as file:
    pickle.dump(cat_model,file)
print("Successfully saved to pickle")


Successfully saved to pickle
